# How Much Accuracy Do You Need?

Original QMCPy demo: [`QMCPy/demos/demo_resume_data/accuracy_and_resume.ipynb`](../../../QMCPy/demos/demo_resume_data/accuracy_and_resume.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/demo_resume_data/accuracy_and_resume.ipynb)

This Julia counterpart preserves the explanatory narrative for the QMCPy resume-data demo. The original notebook depends on QMCPy-specific serialized run artifacts and resume workflows, so the code cells are not translated one-to-one here.


In [1]:
using QMC


The original Python notebook includes code cells that are specific to QMCPy, Python plotting, talk-figure generation, or external tooling. This Julia translation preserves the notebook narrative and mathematical context first, and it should be extended with Julia-native experiments only where there is a clear `QMC.jl` analogue.


## The Problem
Automatic quadrature routines (like those in QMCPy) require the user to specify a target accuracy. But in practice, scientists often don't know what accuracy they need, or their needs may change as they see results or as computational budgets shift.

## The Solution: Resumable Integration in QMCPy
With the `resume` feature in QMCPy, you can start an integration with a loose tolerance, inspect the results, and then resume with a tighter tolerance without starting over.

### How it Works
- **Start with a loose tolerance**: get a quick estimate.
- **Save the computation state**: store integration data to disk.
- **Resume with a tighter tolerance**: continue from where you left off.
- **Repeat as needed**: tighten tolerance iteratively.

This notebook demonstrates the API/workflow and checkpointing behavior. In small examples, timing differences may be dominated by overhead.

Resuming across sessions or machines requires a compatible QMCPy version and compatible solver/problem settings (same integrand definition, dimension, randomization, and stopping criterion family).


## Implementation
The following implementation is carried out in the subclasses of `StoppingCriterion`: 1. A `resume` parameter is added to the `integrate()` method. 2. When `resume=<Data instance>` is supplied, the solver restores the previous state (sample points, transformed values, posterior statistics, etc.) and resumes integration from where it left off.
    * `n_min` is set to the last `n_total` from the resumed run.
    * All relevant internal data (e.g., `ytildefull`, `kappanumap`) are restored from the `Data` object.
3. A `Data.save()` / `Data.load()` API lets you checkpoint integration state to disk so it can be resumed in a future Python session.



Let's see how this works in code. We will use a Genz oscillatory integrand and QMCPy's `CubQMCLatticeG` routine.


### Step 1: Quick Estimate
Suppose you want a quick answer, so you set a loose tolerance.

For this benchmark, we intentionally keep the loose tolerance fairly close to the tight tolerance so the loose run already does meaningful work; this makes the resume incremental benefit easier to observe.

Iteration tracing is enabled by default in the helper cell above. Set `TRACE_ITERATIONS = False` to turn it off.


The `data1` object contains all the diagnostic information from the first integration run. It includes the estimated solution, error bounds, number of samples used, and other useful statistics. This lets you see how close you are to your initial (loose) tolerance and how much work was done so far.


### Step 2: Save the State (Optional)
You can save the integration state to disk for later resumption. If this step is skipped, the workflow continues using the data held in memory.


**Bonus: File Size Optimization with Compression**

QMCPy's `Data.save()` method supports compression to reduce file sizes. This is especially useful for large integration problems or when storage space is limited.  When `compress=True`, the `.gz` extension is automatically appended to maintain consistency with standard compression conventions.

**Example Usage:**

```python
# Save compressed - automatically creates 'data.pkl.gz'
data.save('data.pkl', compress=True, overwrite=True)  

# Load compressed - auto-detection works
loaded_data = Data.load('data.pkl.gz')
```

Let's demonstrate the file size difference and automatic naming behavior.


### Step 3: Resume with Tighter Tolerance
Now suppose you want more accuracy. You can resume from the saved state, using all previous samples.


After resuming with a tighter tolerance, `data2` shows the updated integration state. Compare this to `data1` to see tighter bounds and additional sample usage. This demonstrates state reuse correctness: the resumed run continues from prior work instead of restarting from scratch.


### Step 4: Compare to Starting from Scratch

To demonstrate resume fairly, we report three comparisons:

1. **Incremental cost after the loose run is already done**:
   - Resume: additional work from $N_1$ to $N_2$
   - Fresh: full tight-tolerance run from 0 to $N_2$

2. **New sample count** (less noisy than wall-clock time):
   - Resume adds about $N_2 - N_1$ samples
   - Fresh uses about $N_2$ samples

3. **End-to-end time**:
   - Loose + resume versus fresh tight from scratch

If tight tolerance is known in advance, end-to-end time is often similar (or loose+resume can be slightly slower due to overhead). The practical benefit of resume appears when the loose run has already been computed and you later ask for tighter accuracy.


* The benchmark was successfully tuned to demonstrate that resuming a simulation provides a measurable benefit (speedup > 1) when prior work is available.
* Resuming requires only $524,288$ new samples, significantly fewer than the 1,048,576 samples needed for a fresh run, confirming the efficiency of the workflow.


This fresh-tolerance run is the reference for fair comparisons.

Key interpretation:

- The resume workflow should not be interpreted as a faster way to compute a known tight tolerance from the beginning. If the tight tolerance requirement is known in advance, a fresh tight run is usually just as efficient or slightly more efficient (since saving and loading data take time).

- The benefit of resume appears when the loose run has already been performed in experiments. At that point, the loose samples are sunk computational cost. Resuming avoids discarding them and evaluates only the additional samples needed to meet the tighter tolerance.


## Conclusion

- With the resume feature, you can adaptively decide how much accuracy you need, and only pay for more if you want it.
- You can pause, checkpoint, and resume long computations.
- This is a practical answer to Lyness's Case B and Case C: you don't have to know your accuracy in advance!

Try it yourself: change the tolerances, or resume from a saved file in a new session.
